In [0]:
from pyspark.sql.functions import *

df = spark.table("bmw_vehicle_intelligence.silver.bmw_vehicle")

print("Total Records:", df.count())

display(df.limit(5))

In [0]:
from pyspark.sql.functions import *

df = spark.table("bmw_vehicle_intelligence.silver.bmw_vehicle")

df.select(
    max("vehicle_id").alias("latest_vehicle_id")
).show()

In [0]:
new_data = [
    (10001, 5001, "BMWNEW000001", "BMW X5", 2026, "DIESEL", "2026-11-20"),
    (10002, 5002, "BMWNEW000002", "BMW X3", 2026, "PETROL", "2026-11-21"),
    (10003, 5003, "BMWNEW000003", "BMW i4", 2026, "ELECTRIC", "2026-11-22")
]

columns = [
    "vehicle_id", "customer_id", "vin",
    "model", "model_year", "fuel_type", "purchase_date"
]

new_df = spark.createDataFrame(new_data, columns)

display(new_df)

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table(
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)

latest_id = silver_df.select(
    max("vehicle_id")
).first()[0]
source_df = (
    spark.read
    .option("header", "false")
    .csv("/Volumes/bmw_vehicle_intelligence/bronze/raw_data/bmw_vehicles")
)
source_df = source_df.toDF(
    "vehicle_id", "customer_id", "vin",
    "model", "model_year", "fuel_type", "purchase_date"
)
incremental_df = source_df.filter(
    col("vehicle_id").cast("int") > latest_id
)
print("Latest Loaded ID:", latest_id)
print("New Records:", incremental_df.count())

display(incremental_df)

In [0]:
print("New DataFrame Count:", new_df.count())

display(new_df)

In [0]:
from pyspark.sql.functions import *
incremental_df = (
    new_df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", to_date("purchase_date"))
    .withColumn("vin", upper(trim(col("vin"))))
    .withColumn("model", trim(col("model")))
    .withColumn("fuel_type", upper(trim(col("fuel_type"))))
    .withColumn("processed_at", current_timestamp())
)
incremental_df.write.format("delta").mode("append").saveAsTable(
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)

print("Incremental loading completed!")

In [0]:
df = spark.table(
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)

print("Total Records:", df.count())

display(
    df.filter(col("vehicle_id") >= 10001)
)

In [0]:
merge_data = [
    (10001, 5001, "BMWNEW000001", "BMW X5 UPDATED", 2026, "DIESEL", "2026-11-20"),
    (10004, 5004, "BMWNEW000004", "BMW X7", 2026, "PETROL", "2026-11-23")
]

columns = [
    "vehicle_id",
    "customer_id",
    "vin",
    "model",
    "model_year",
    "fuel_type",
    "purchase_date"
]

merge_df = spark.createDataFrame(merge_data, columns)

display(merge_df)